# 06 – Seltene Klassen, Gewichtung und SMOTE

**Lernziele:** Accuracy kritisch lesen, Alternativen vergleichen und Resampling ausschließlich im Training durchführen. Dauer: ca. 35 Minuten.

SMOTE gehört zu **imbalanced-learn**, einer Ergänzung mit scikit-learn-kompatibler API. Installiere die aktualisierte `requirements.txt`; alternativ im Kernel `%pip install "imbalanced-learn>=0.14,<0.15"`, anschließend gegebenenfalls Kernel neu starten.

Wir simulieren unabhängige numerische Qualitätsmessungen mit seltenen Fehlern. Das ist kein Nachweis für den Nutzen von SMOTE auf Produktionsdaten.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.pipeline import Pipeline


In [ ]:
X, y = make_classification(n_samples=1800, n_features=8, n_informative=4,
                           n_redundant=2, weights=[0.96, 0.04], flip_y=0.01, random_state=42)
X_entw, X_test, y_entw, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
display(pd.Series(y_entw).value_counts().rename('Entwicklungsdaten'))


## 1. Warum Accuracy allein täuscht

Fast immer „kein Fehler“ zu sagen kann hohe Accuracy liefern. Recall misst gefundene echte Fehler, Precision den Anteil echter Fehler unter den Alarmen. Balanced Accuracy gewichtet beide Klassen gleich. Average Precision bewertet Scores über Schwellen hinweg; ihre Referenz hängt von der Häufigkeit positiver Fälle ab.

## 2. Gleiche Folds, mehrere Strategien

Mehrheitsbaseline, kein Ausgleich, Klassengewichte, zufälliges Duplizieren und SMOTE werden verglichen. SMOTE interpoliert zwischen benachbarten Minderheitsfällen. Erst skalieren, damit große Einheiten nicht die Nachbarschaft dominieren. Jeder Trainingsfold braucht mindestens `k_neighbors + 1` Minderheitsfälle.


In [ ]:
def modell(sampler='passthrough', gewicht=None):
    return Pipeline([('skalierung', StandardScaler()), ('sampling', sampler),
                     ('modell', LogisticRegression(class_weight=gewicht, max_iter=1000))])
modelle = {
    'Mehrheitsbaseline': DummyClassifier(strategy='most_frequent'),
    'Ohne Ausgleich': modell(),
    'Klassengewichte': modell(gewicht='balanced'),
    'RandomOverSampler': modell(RandomOverSampler(random_state=42)),
    'SMOTE': modell(SMOTE(k_neighbors=3, random_state=42)),
}
folds = list(StratifiedKFold(n_splits=4, shuffle=True, random_state=42).split(X_entw, y_entw))
assert min(np.bincount(y_entw[tr]).min() for tr, va in folds) > 3
zeilen = []
for name, kandidat in modelle.items():
    result = cross_validate(kandidat, X_entw, y_entw, cv=folds,
                            scoring={'accuracy': 'accuracy', 'bal_acc': 'balanced_accuracy', 'ap': 'average_precision', 'recall': 'recall'}, error_score='raise')
    zeilen.append({'Methode': name, **{k[5:]: v.mean() for k, v in result.items() if k.startswith('test_')}})
vergleich = pd.DataFrame(zeilen).set_index('Methode')
display(vergleich.round(3))
vergleich[['accuracy', 'bal_acc', 'ap', 'recall']].plot.bar(figsize=(10, 4), ylim=(0, 1))
plt.xticks(rotation=20); plt.tight_layout(); plt.show()


Die **imblearn-Pipeline** resampelt nur während `fit`. Validierungs- und Testdaten behalten ihre ursprüngliche Verteilung. Die normale sklearn-Pipeline unterstützt diesen Sampling-Schritt nicht.

Niemals `SMOTE().fit_resample(X, y)` vor Split oder CV: Synthetische Trainingspunkte könnten Informationen aus späteren Prüfdaten enthalten. Einen Testsatz auszugleichen würde zusätzlich eine andere Verteilung bewerten.

## 3. Nach CV auswählen und final bewerten

Hier gilt Average Precision vorab als Auswahlkriterium. Im Einsatz braucht es zusätzlich Fehlalarmkosten und Mindest-Recall. Eine Entscheidungsschwelle wird ausschließlich auf Entwicklungs-/Validierungsdaten gewählt; hier bleibt die Standardschwelle unverändert.


In [ ]:
bester_name = vergleich['ap'].idxmax()
bestes = modelle[bester_name].fit(X_entw, y_entw)
vorhersage = bestes.predict(X_test)
print('Aus CV gewählt:', bester_name)
print(classification_report(y_test, vorhersage, zero_division=0))
print('Konfusionsmatrix (Zeilen wahr, Spalten vorhergesagt):')
print(confusion_matrix(y_test, vorhersage))
print('Test Average Precision:', round(average_precision_score(y_test, bestes.predict_proba(X_test)[:, 1]), 3))
print('Unveränderter Testumfang:', len(y_test), 'Positivanteil:', round(y_test.mean(), 3))


## 4. Grenzen und Alternativen

- SMOTE kann Rauschen, Ausreißer und Klassenüberlappungen verstärken. Undersampling spart Aufwand, verliert aber Mehrheitsinformationen.
- Gewöhnliches SMOTE auf Kategoriecodes oder One-Hot-Spalten ist für gemischte Daten ungeeignet. `SMOTENC` berücksichtigt Kategorien, braucht aber eine passende Verarbeitungskette.
- Synthetische Messungen können physikalisch unmöglich sein. Bei Zeit oder Gruppen zunächst korrekt trennen und auch innerhalb der Trainingsfolds die Plausibilität prüfen.
- Nach Oversampling sind Wahrscheinlichkeiten nicht automatisch an die reale Klassenhäufigkeit angepasst. Kalibrierung und Schwellenwahl benötigen unveränderte Validierungsdaten.

**Mini-Aufgabe:** Welche Methode gewinnt bei Recall, welche bei Average Precision? Warum muss das nicht dieselbe sein? Weiter: Übung 05.

Quellen: [Leakage vermeiden](https://imbalanced-learn.org/stable/common_pitfalls.html), [SMOTE](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html), [imblearn-Pipeline](https://imbalanced-learn.org/stable/references/generated/imblearn.pipeline.Pipeline.html).
